# Infrastructure

The parts of a screening campaign that are not physics: what unit a number is
in, how to survive a walltime limit, how to process a corpus that does not fit
in memory, and how to hand the survivors to a real DFT code.

None of this is interesting. All of it is what separates a calculation you ran
once from a campaign that finished.

In [1]:
import matverse as mv
import numpy as np
import pandas as pd

mv.pl.set_style()

🔬 Starting plot initialization...
🧪 Calculators available: 6
    • emt — EMT (LGPL-2.1)
    • lj — Lennard-Jones (LGPL-2.1)
    • mace-mpa — mace-mpa (unstated)
    • mace-omat — mace-omat (unstated)
    • sevennet — sevennet (unstated)
    • chgnet — chgnet (unstated)
🖥️ NVIDIA CUDA GPUs: 1
    • [CUDA 0] NVIDIA H100 80GB HBM3 — 79.1 GB, compute 9.0

                   __
   ____ ___  ____ _/ /__   _____  _____________
  / __ `__ \/ __ `/ __/ | / / _ \/ ___/ ___/ _ \
 / / / / / / /_/ / /_ | |/ /  __/ /  (__  )  __/
/_/ /_/ /_/\__,_/\__/ |___/\___/_/  /____/\___/

🔖 Version: 0.1.16   🧮 Functions: 153   📚 Tutorials: https://matverse.readthedocs.io/
✅ set_style complete.



In [2]:
md = mv.datasets.metals()
mv.pp.describe(md)
mv.calc.energy(md, level="emt")

md.obs[["name", "energy_per_atom_emt"]].round(4)

,name,energy_per_atom_emt
0,Al,-0.0016
1,Cu,-0.0050
2,Ni,-0.0076
3,Ag,0.0010
4,Au,0.0022
5,Pd,0.0004
6,Pt,-0.0001


## What is this object carrying?

`mv.utils.summary` is the human-readable inventory — every axis, every variant,
every level, in one string.

In [3]:
print(mv.utils.summary(md))

matverse dataset: 7 materials x 7 elements
  elements   Al, Ni, Cu, Pd, Ag, Pt, Au
  structures input
  levels
    emt              EMT [LGPL-2.1]
  provenance 3 operations
    data.from_structures
    pp.describe(source='input')
    calc.energy(level='emt', source='input')


## Units

A number without a unit is not a result. matverse works internally in eV and
ångström, records the unit of every column it writes, and converts on request
rather than silently.

In [4]:
mv.utils.INTERNAL_UNITS

{'energy': 'eV',
 'energy_per_atom': 'eV/atom',
 'e_above_hull': 'eV/atom',
 'formation_energy': 'eV/atom',
 'max_force': 'eV/angstrom',
 'force_magnitude': 'eV/angstrom',
 'volume': 'angstrom^3',
 'density': 'g/cm^3',
 'min_distance': 'angstrom',
 'bulk_modulus': 'GPa',
 'shear_modulus': 'GPa'}

In [5]:
mv.utils.check_units(md)

{'lattice_parameter': None,
 'nsites': None,
 'n_elements': None,
 'molecular_weight': None,
 'volume': 'angstrom^3',
 'density': 'g/cm^3',
 'volume_per_atom': 'angstrom^3',
 'is_periodic': None,
 'energy_emt': 'eV',
 'energy_per_atom_emt': 'eV/atom'}

Every column matverse wrote has a unit; `lattice_parameter` and `nsites` come
back `None` because nothing declared one. A column you compute yourself is in
the same position until you say otherwise.

In [6]:
md.obs["cohesive_estimate"] = -md.obs["energy_per_atom_emt"].to_numpy(dtype=float)
mv.utils.set_units(md, "cohesive_estimate", "eV/atom")

mv.utils.check_units(md)["cohesive_estimate"]

'eV/atom'

`mv.utils.convert` goes the other way — it takes a column in *someone else's*
unit and brings it into matverse's. That is the direction that matters, because
the numbers arriving from a collaborator or a paper are the ones in kJ/mol.

Simulate that: build a column in kJ/mol, declare it, and convert it back.

In [7]:
md.obs["reported_energy"] = (
    md.obs["energy_per_atom_emt"].to_numpy(dtype=float) / mv.utils.TO_EV["kj/mol"])
mv.utils.set_units(md, "reported_energy", "kJ/mol")

mv.utils.convert(md, "reported_energy", kind="energy")
md.obs[["name", "reported_energy", "reported_energy_ev",
        "energy_per_atom_emt"]].round(4)

,name,reported_energy,reported_energy_ev,energy_per_atom_emt
0,Al,-0.1507,-0.0016,-0.0016
1,Cu,-0.4778,-0.0050,-0.0050
2,Ni,-0.7325,-0.0076,-0.0076
3,Ag,0.0924,0.0010,0.0010
4,Au,0.2135,0.0022,0.0022
5,Pd,0.0405,0.0004,0.0004
6,Pt,-0.0102,-0.0001,-0.0001


`reported_energy_ev` and `energy_per_atom_emt` agree, which is the check worth
running: a unit conversion that does not round-trip is a unit conversion you
should not trust.

Note that it **deposits a new column** rather than rewriting the old one. A
converted column sitting beside its original is auditable; a silently rewritten
one is the bug this function exists to prevent, one step later. The declared
unit is carried along.

In [8]:
mv.utils.TO_EV["kj/mol"], mv.utils.TO_ANGSTROM["bohr"]

(0.010364269656262175, 0.529177210903)

## Surviving a walltime limit

A screen that takes six hours on a queue with a four-hour limit needs to be
resumable, and the object is where that state lives.

`mv.utils.checkpoint` writes the object to disk with a note.

In [9]:
import tempfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
path = mv.utils.checkpoint(md, workdir / "screen.h5ad", note="after emt")
Path(path).name, round(Path(path).stat().st_size / 1e3, 1)

('screen.h5ad', 76.5)

`mv.utils.resume` answers the question a restarted job actually asks: *which
rows still need doing?*

In [10]:
import anndata

reloaded = anndata.read_h5ad(path)
reloaded.obs.loc[reloaded.obs_names[:3], "band_gap_pbe"] = [1.2, 0.0, 2.4]

todo = mv.utils.resume(reloaded, "band_gap_pbe")
todo, todo.sum()

(array([False, False, False,  True,  True,  True,  True]), np.int64(4))

Three rows were done before the job died; four remain. Note that this works on
the **reloaded** object — an object you cannot pick back up is not a
checkpoint.

```{note}
That sentence is load-bearing. h5ad stores `uns['provenance']` as a list and
reads it back as a numpy array, so until v0.1.13 every operation on a reloaded
object failed on its own provenance write. Saving is only useful if the object
can be worked on afterwards, and there is now a test that says so.
```

## Corpora larger than memory

`mv.utils.chunks` slices the object into pieces, yielding the starting row
index with each — so a piece can always be written back to where it came
from.

In [11]:
for start, piece in mv.utils.chunks(md, size=3):
    print(f"rows {start}-{start + piece.n_obs - 1}: {list(piece.obs['name'])}")

rows 0-2: ['Al', 'Cu', 'Ni']
rows 3-5: ['Ag', 'Au', 'Pd']
rows 6-6: ['Pt']


`map_chunks` runs an operation over each piece, optionally checkpointing as it
goes, and can skip rows already done — which is the resumable-screen pattern in
one call.

In [12]:
fresh = mv.datasets.metals()
mv.pp.describe(fresh)

report = mv.utils.map_chunks(
    fresh,
    lambda piece: mv.calc.energy(piece, level="emt"),
    size=3,
    checkpoint_to=workdir / "progress.h5ad",
)
report

{'size': 3, 'n_processed': 7, 'n_skipped': 0, 'errors': []}

In [13]:
fresh.obs[["name", "energy_per_atom_emt"]].round(4)

,name,energy_per_atom_emt
0,Al,-0.0016
1,Cu,-0.0050
2,Ni,-0.0076
3,Ag,0.0010
4,Au,0.0022
5,Pd,0.0004
6,Pt,-0.0001


The results landed on the parent object even though the work happened
chunk-by-chunk, and the checkpoint on disk is current.

## Getting onto a cluster

`mv.utils.slurm_script` writes a submission script rather than submitting
anything. Submitting is the scheduler's job and every site does it differently;
generating a correct script is the part that is the same everywhere.

In [14]:
script = mv.utils.slurm_script(
    "python screen.py --checkpoint screen.h5ad",
    path=workdir / "screen.sbatch",
    partition="normal", hours=8, cpus=16, memory="64GB",
    job_name="alni-screen",
    setup="module load python/3.12\nsource $SCRATCH/env/bin/activate",
)
print(Path(script).read_text())

#!/bin/bash
#SBATCH --job-name=alni-screen
#SBATCH --partition=normal
#SBATCH --time=08:00:00
#SBATCH --cpus-per-task=16
#SBATCH --mem=64GB
#SBATCH --output=%x-%j.out

set -euo pipefail

# Caches default to $HOME, which is small and shared. Point them at
# scratch so a model download does not fill a home directory.
export HF_HOME="${SCRATCH:-$PWD}/hf"
export XDG_CACHE_HOME="${SCRATCH:-$PWD}/cache"
export PIP_CACHE_DIR="${SCRATCH:-$PWD}/pip"

module load python/3.12
source $SCRATCH/env/bin/activate

python python screen.py --checkpoint screen.h5ad



### Submitting it

`mv.utils.submit` shells out to `sbatch` and records the job id **on the
object**, so "which job is computing this dataset" is answerable from the data
rather than from shell history.

In [15]:
entry = mv.utils.submit(md, script, dry_run=True)
entry

{'script': '/tmp/tmpygnvau_3/screen.sbatch',
 'command': 'sbatch /tmp/tmpygnvau_3/screen.sbatch',
 'job_id': None,
 'state': 'dry run'}

`dry_run=True` returns the command without running it, which is what you want
on a login node — and what this page uses, because a documentation build should
not queue jobs.

Submissions accumulate through the same record machinery as campaign rounds, so
they survive `write_h5ad`: a list of dicts in `uns` does not.

In [16]:
mv.records(md.uns["submissions"], "jobs")

[{'script': '/tmp/tmpygnvau_3/screen.sbatch',
  'command': 'sbatch /tmp/tmpygnvau_3/screen.sbatch',
  'job_id': None,
  'state': 'dry run'}]

In [17]:
mv.utils.job_status(md)

{'n_jobs': 1,
 'states': {},
 'note': 'nothing was submitted for real; check for dry runs'}

With real submissions that reads `squeue`, and falls back to `sacct` for jobs
that have finished — because `squeue` forgets a job shortly after it ends,
which is the usual reason a hand-rolled poll reports a completed job as
missing.

matverse stops there. It is not a workflow manager and does not retry, chain or
monitor; atomate2, quacc and AiiDA do that, and a fourth would be a maintenance
liability. What it adds is the link back to the dataset.

```{warning}
It writes `--time`, `--mem`, `--cpus-per-task` and `--partition` explicitly and
does not rely on defaults. Slurm's defaults are typically one CPU, a few
hundred megabytes and a short walltime, so a job submitted without them fails
in a way that looks like a bug in your code.
```

## Handing over to DFT

EMT and machine-learned potentials narrow the field. The shortlist gets real
DFT, and matverse's job is to generate inputs and harvest results — not to run
VASP, which the queue does.

In [18]:
mv.dft.presets()

{'relax': {'input_set': 'pymatgen.io.vasp.sets.MPRelaxSet',
  'reference': 'PBE+U',
  'description': "Materials Project relaxation — PBE(+U), the settings MP's own entries use, so results are comparable with the hull."},
 'static': {'input_set': 'pymatgen.io.vasp.sets.MPStaticSet',
  'reference': 'PBE+U',
  'description': 'Single point on a fixed geometry, denser k-mesh than relax.'},
 'bands': {'input_set': 'pymatgen.io.vasp.sets.MPNonSCFSet',
  'reference': 'PBE+U',
  'description': 'Non-self-consistent run along a k-path, for a band structure.'},
 'scan': {'input_set': 'pymatgen.io.vasp.sets.MPScanRelaxSet',
  'reference': 'r2SCAN',
  'description': 'r2SCAN relaxation. A different level of theory from PBE — tag it as one.'},
 'hse': {'input_set': 'pymatgen.io.vasp.sets.MPHSERelaxSet',
  'reference': 'HSE06',
  'description': 'HSE06 hybrid. Expensive; screen with something cheaper first.'}}

In [19]:
shortlist = md[md.obs["name"].isin(["Cu", "Al"])].copy()
written = mv.dft.write_inputs(shortlist, workdir / "runs", code="vasp",
                              preset="relax")
[Path(p).name for p in written]

['0', '1']

In [20]:
sorted(p.name for p in Path(written[0]).iterdir())

['INCAR', 'KPOINTS', 'POSCAR', 'POTCAR.spec', 'matverse.json']

INCAR, POSCAR, KPOINTS and a POTCAR *specification* — the last is a list of
which potentials to concatenate rather than the potentials themselves, because
those are licensed and cannot be redistributed.

### Where did the jobs get to?

In [21]:
mv.dft.status(shortlist, workdir / "runs")

{'root': '/tmp/tmpygnvau_3/runs',
 'n_total': 2,
 'n_finished': 0,
 'n_missing': 2,
 'missing': ['0', '1'],
 'truncated': False}

Nothing has run, so both are missing. Pretend one finished:

In [22]:
(Path(written[0]) / "vasprun.xml").write_text("<modeling/>")
mv.dft.status(shortlist, workdir / "runs")

{'root': '/tmp/tmpygnvau_3/runs',
 'n_total': 2,
 'n_finished': 1,
 'n_missing': 1,
 'missing': ['1'],
 'truncated': False}

```{note}
`status` resolves runs through a **manifest** written next to the inputs, not by
matching directory names. A workflow manager that renames `run-000` to
`job-41725` would otherwise silently attach results to the wrong row — which is
the usual way a hand-rolled harvest goes wrong, and it goes wrong quietly.
```

### Harvesting

`read_outputs` parses whatever finished and deposits it at the level you name.

In [23]:
mv.dft.read_outputs(shortlist, workdir / "runs", level="pbe")

shortlist.obs[["name", "energy_pbe", "dft_error_pbe"]]

,name,energy_pbe,dft_error_pbe
0,Al,NaN,AttributeError: 'Vasprun' object has no attrib...
1,Cu,NaN,no output found


Both are NaN with a reason, because the `<modeling/>` above is not a real
vasprun. **That is the designed behaviour**: a run that failed becomes a NaN
carrying an explanation, not a dropped row.

A dropped row is how a screen quietly becomes a screen over the subset that
happened to converge — and the bias that introduces points exactly the wrong
way, since the calculations that fail are the difficult, interesting ones.

In [24]:
mv.dft.read_dos(shortlist, workdir / "runs", level="pbe")

[k for k in shortlist.obs if k.endswith("_pbe")]

['energy_pbe',
 'energy_per_atom_pbe',
 'band_gap_pbe',
 'converged_pbe',
 'dft_error_pbe',
 'is_direct_gap_pbe',
 'vbm_pbe',
 'cbm_pbe',
 'fermi_level_pbe',
 'dos_at_fermi_pbe',
 'is_metal_pbe']

`read_dos` fills the density of states onto the shared energy grid, and derives
the band gap and Fermi level from it. Same story: absent runs are recorded as
absent.

With real output files, `obsm['dos_pbe']` would plot with `mv.pl.spectra` like
any other curve, and the gap would be an ordinary `obs` column that
`mv.screen.filter` can reach.

## Licences

A last piece of bookkeeping that is easy to skip and expensive to get wrong.
Levels of theory carry their licence, so the question "can this result go in a
commercial report" has an answer in the object.

In [25]:
mv.calc.check_licenses(md)

[]

In [26]:
mv.check_commercial_use(md)

[]

```{seealso}
[Getting data in and out](data_io.ipynb) is the other end of the plumbing.
[Scale and first principles](scale_and_dft.md) covers the same ground in prose,
with more on the failure modes of very large corpora.
```